## GPT2 implementation


Tokenizer


In [ ]:
class BPETokenizer(nn.Module:

KV cache


In [ ]:
class Cache:

	def __init__(self, n, max_seq_len, h)

Normalization techniques

- batchnorm
- layernorm
- instancenorm
- groupnorm
- RMSnorm
- pre- or post-LayerNorm


In [ ]:
class RMSNorm(nn.Module): 

	def __init__(self, d_embed:int, eps:float=1e-4): 
		
		super().__init__()
		self.eps = eps
		
		self.weight = nn.Parameter(d_embed)

	def forward(self, x_in: torch.Tensor): 

		variance = x_in.pow(2).mean(dim=-1, keepdim=True) # mean sum of squares along feature dimension, (B, T, D)

		x_rms = torch.rsqrt(variance + self.eps) # (B, T, D)

		return x_in * x_rms * self.weight

class LayerNorm(nn.Module):

Residual connections

- implementing a residual stream
- implementing ByteDance HC
- implementing Deepseek mHC


Learning rate scheduler

- cosine decay


Optimizer

- AdamW
- MUON
- SHAMPOO


Dropout

_dropout as ensemble learning_

- dropout acts as ensemble learning during inference. During training time, say we set dropout = 0.99, thus we train lots of different sub-networks, each lighting up a different set of neurons. Each of these neurons learns to produce the output, independent of each other, or if there's neurons shared between subnetworks, the neuron becomes a shared parameter, but this weakens the ensemble learning since if there's a lot of shared neurons, your subnetworks are not producing meaningfully different outputs.
- at inference time, we use .eval() to turn off dropout. Now each subnetwork is activated in the forward pass, and the activations are a combination of all the subnetworks votes on what the right activation is, since each has learned a different 'correct answer' for the output vector.

_dropout during self-attention_

- we place dropout in 2 different places
  - after softmax produces the scores
  - right before we merge with the layer output with the residual stream
- why there?
  - we don't want to dropout the residual stream, rather we care about the weights in the layer, learning to develop subnetworks. The residual stream changes on every token - this is not something the model needs to learn

  - models can become overly reliant and attend only to the tokens immediately adjacent to the next token (e.g. the cat chases the dog. It was very fast. It refers to the \_**\_ -> the model may learn, when predicting the \_\_** to attend only to the closest tokens, so 'dog' when 'it' may refer to the cat)


Implement GPT2 from scratch
